# Python Statistical Analysis in Practice — 02: Descriptive Statistics & EDA

*Author: Jason JJ Li · Peking University Institute of Population Research*

Before fitting any model, you must **look at your data**. This lecture builds a
systematic toolkit for Exploratory Data Analysis (EDA) — the process of
summarising, visualising, and understanding a dataset before any formal inference.

**What you will learn today:**

1. Central tendency: mean, median, mode — and when they disagree
2. Spread: variance, standard deviation, IQR, coefficient of variation
3. Distribution shape: skewness and kurtosis
4. The EDA visualisation toolkit: histogram, KDE, boxplot, violin, Q-Q plot
5. Outlier detection: z-score and IQR methods
6. Why you must always plot — Anscombe's Quartet
7. Bivariate EDA: scatter plots and correlation (Pearson vs Spearman)
8. Multivariate overview: pair plots and heatmaps
9. The full EDA pipeline — applied to a planted dataset

**The simulation bridge:** we generate every dataset with known parameters,
run EDA, and confirm that what we find matches what we planted.


---

## Part 1: Setup


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 3)
pd.set_option('display.float_format', '{:.3f}'.format)

COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#A23B72',
    'accent':    '#F18F01',
    'success':   '#27AE60',
    'danger':    '#E74C3C',
    'neutral':   '#95A5A6',
}
RNG = np.random.default_rng(42)

import scipy
print('Libraries loaded.')
print(f'NumPy {np.__version__}  |  Pandas {pd.__version__}  |  Scipy {scipy.__version__}')


---

## Part 2: Central Tendency — Mean, Median, and Mode

The three measures of central tendency each answer a slightly different question:

| Measure | What it measures | Best used when |
|---|---|---|
| **Mean** | Arithmetic average — sum / n | Data is roughly symmetric, no extreme values |
| **Median** | Middle value (50th percentile) | Data is skewed, or outliers are present |
| **Mode** | Most frequent value | Categorical data, or multimodal distributions |

### 2.1 When Mean ≈ Median: Symmetric Data

For a symmetric distribution, all three measures cluster together.


In [ ]:
# Symmetric normal distribution
sym_data = RNG.normal(loc=100, scale=15, size=2000)

mean_val   = sym_data.mean()
median_val = np.median(sym_data)
mode_val   = stats.mode(sym_data.round(0), keepdims=True).mode[0]

print('=== Symmetric distribution (Normal, μ=100, σ=15) ===')
print(f'  Mean   : {mean_val:.3f}')
print(f'  Median : {median_val:.3f}')
print(f'  Mode   : {mode_val:.1f}  (rounded to integers)')
print(f'  Mean - Median = {mean_val - median_val:.3f}  ← close to 0 for symmetric data')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sym_data, bins=50, density=True, color=COLORS['primary'],
        edgecolor='white', alpha=0.8, label='Data')
ax.axvline(mean_val,   color=COLORS['danger'],  linestyle='-',  lw=2.5, label=f'Mean={mean_val:.1f}')
ax.axvline(median_val, color=COLORS['success'], linestyle='--', lw=2.5, label=f'Median={median_val:.1f}')
ax.axvline(mode_val,   color=COLORS['accent'],  linestyle=':',  lw=2.5, label=f'Mode≈{mode_val:.0f}')
ax.set_xlabel('Value'); ax.set_ylabel('Density')
ax.set_title('Symmetric data: Mean ≈ Median ≈ Mode')
ax.legend()
plt.tight_layout()
plt.show()


### 2.2 When Mean ≠ Median: Right-Skewed Data

Right-skewed distributions — income, house prices, hospital stay durations — have a
long right tail that **pulls the mean upward** but leaves the median unaffected.

> **Rule of thumb:** if Mean > Median, the distribution is right-skewed.
> Report the **median** (and IQR) as the primary summary, not the mean.


In [ ]:
# Simulate a right-skewed income-like distribution
# Log-normal: log(X) ~ Normal so X itself is right-skewed
income_data = RNG.lognormal(mean=10.5, sigma=0.8, size=2000)  # income in yuan

mean_inc   = income_data.mean()
median_inc = np.median(income_data)
mode_inc   = stats.mode(income_data.round(-3), keepdims=True).mode[0]

print('=== Right-skewed distribution (simulated income) ===')
print(f'  Mean   : ¥{mean_inc:,.0f}   ← pulled up by high earners')
print(f'  Median : ¥{median_inc:,.0f}   ← a more typical value')
print(f'  Mode   : ¥{mode_inc:,.0f}   (rounded to nearest 1000)')
print(f'  Mean / Median = {mean_inc/median_inc:.2f}x  ← mean is {(mean_inc/median_inc-1)*100:.0f}% above median')
print()
print('  → For income data, official statistics use MEDIAN income.')
print('    A small number of very high earners inflates the mean.')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Right-Skewed Income Distribution — Mean vs Median', fontsize=12, fontweight='bold')

# Left: full range
ax1.hist(income_data, bins=80, density=True, color=COLORS['primary'],
         edgecolor='white', alpha=0.8)
ax1.axvline(mean_inc,   color=COLORS['danger'],  lw=2.5, linestyle='-',  label=f'Mean=¥{mean_inc:,.0f}')
ax1.axvline(median_inc, color=COLORS['success'], lw=2.5, linestyle='--', label=f'Median=¥{median_inc:,.0f}')
ax1.set_xlabel('Income (¥)'); ax1.set_ylabel('Density')
ax1.set_title('Full distribution (note the long right tail)')
ax1.legend()

# Right: trimmed to p95 for better view
p95 = np.percentile(income_data, 95)
trimmed = income_data[income_data <= p95]
ax2.hist(trimmed, bins=60, density=True, color=COLORS['secondary'],
         edgecolor='white', alpha=0.8, label=f'≤ p95 (¥{p95:,.0f})')
ax2.axvline(mean_inc,   color=COLORS['danger'],  lw=2.5, linestyle='-',  label=f'Mean=¥{mean_inc:,.0f}')
ax2.axvline(median_inc, color=COLORS['success'], lw=2.5, linestyle='--', label=f'Median=¥{median_inc:,.0f}')
ax2.set_xlabel('Income (¥)'); ax2.set_ylabel('Density')
ax2.set_title('Trimmed to 95th percentile (zoom in)')
ax2.legend()
plt.tight_layout()
plt.show()


### 2.3 Bimodal Distributions: Mode Reveals What Mean and Median Hide

If a dataset has two peaks (bimodal), the mean and median both land *between* the
peaks — a value that no one actually has. The mode is the only measure that
captures multi-peakedness.


In [ ]:
# Simulate a bimodal distribution: two groups mixed together
group_A = RNG.normal(loc=160, scale=6, size=800)   # e.g. female heights
group_B = RNG.normal(loc=174, scale=7, size=800)   # e.g. male heights
mixed   = np.concatenate([group_A, group_B])

print('=== Bimodal distribution (mixed heights) ===')
print(f'  Mean   : {mixed.mean():.1f} cm  ← between both groups')
print(f'  Median : {np.median(mixed):.1f} cm  ← between both groups')
print()
print('  → Neither mean nor median tells you there are TWO sub-populations.')
print('    Always visualise!')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Bimodal Distribution: The Mean Lies in No-Man's Land", fontsize=12, fontweight='bold')

# Left: mixed, coloured by true group
axes[0].hist(group_A, bins=40, density=True, color=COLORS['primary'],   alpha=0.7, edgecolor='white', label='Group A (μ=160)')
axes[0].hist(group_B, bins=40, density=True, color=COLORS['secondary'], alpha=0.7, edgecolor='white', label='Group B (μ=174)')
axes[0].axvline(mixed.mean(), color=COLORS['danger'], lw=2.5, linestyle='--', label=f'Overall mean={mixed.mean():.1f}')
axes[0].set_xlabel('Height (cm)'); axes[0].set_ylabel('Density')
axes[0].set_title('True group membership revealed')
axes[0].legend()

# Right: same data, group unknown
axes[1].hist(mixed, bins=60, density=True, color=COLORS['neutral'], alpha=0.8, edgecolor='white', label='All data')
axes[1].axvline(mixed.mean(),       color=COLORS['danger'],  lw=2.5, linestyle='-',  label=f'Mean={mixed.mean():.1f}')
axes[1].axvline(np.median(mixed),   color=COLORS['success'], lw=2.5, linestyle='--', label=f'Median={np.median(mixed):.1f}')
axes[1].set_xlabel('Height (cm)'); axes[1].set_ylabel('Density')
axes[1].set_title('Without group labels — looks like one blob')
axes[1].legend()
plt.tight_layout()
plt.show()


---

## Part 3: Spread — How Variable Is the Data?

Knowing the centre is not enough. Two datasets can have the exact same mean but
completely different spreads. Four measures of spread, each with a different use:

| Measure | Formula | Sensitive to outliers? | Typical use |
|---|---|---|---|
| **Range** | max − min | Very sensitive | Quick first look only |
| **Variance** | $\frac{1}{n-1}\sum(x_i - \bar{x})^2$ | Yes (squared deviations) | Mathematical basis for many tests |
| **Std deviation** | $\sqrt{\text{Variance}}$ | Yes | Same unit as data; most common |
| **IQR** | Q3 − Q1 | No | Robust; always report with median |

### 3.1 Variance and Standard Deviation: The Maths Behind Them


In [ ]:
# Verify that manual formula matches numpy
sample = np.array([12.0, 14.0, 11.0, 15.0, 13.0, 20.0, 12.5, 14.5, 13.5, 11.5])

# Manual calculation
deviations   = sample - sample.mean()
sq_devations = deviations ** 2
variance_manual = sq_devations.sum() / (len(sample) - 1)  # n-1 = Bessel's correction
std_manual      = np.sqrt(variance_manual)

print('=== Manual vs NumPy ===')
print(f'  Values     : {sample.tolist()}')
print(f'  Mean       : {sample.mean():.4f}')
print(f'  Deviations : {deviations.round(2).tolist()}')
print(f'  Sq. devs   : {sq_devations.round(2).tolist()}')
print()
print(f'  Variance (manual) : {variance_manual:.4f}')
print(f'  Variance (numpy)  : {sample.var(ddof=1):.4f}    ← ddof=1 uses n-1')
print(f'  Std (manual)      : {std_manual:.4f}')
print(f'  Std (numpy)       : {sample.std(ddof=1):.4f}')
print()
print("  Why n-1 (Bessel's correction)?")
print('  We estimated the mean from the same sample, which slightly underestimates')
print('  true variance. Dividing by n-1 corrects this bias.')


### 3.2 IQR: Robust to Outliers

The interquartile range (IQR = Q3 − Q1) spans the middle 50% of the data.
It is completely unaffected by extreme values, making it the paired companion
to the median.


In [ ]:
# Demonstrate std vs IQR robustness
base_data  = RNG.normal(loc=50, scale=5, size=200)
with_outlier = np.append(base_data, [150, 160, 155])  # inject 3 extreme outliers

def spread_summary(x, label):
    q1, q3 = np.percentile(x, [25, 75])
    print(f'{label}:')
    print(f'  n={len(x)}, mean={x.mean():.2f}, median={np.median(x):.2f}')
    print(f'  Std={x.std(ddof=1):.2f}  |  IQR={q3-q1:.2f}  |  Range=[{x.min():.1f}, {x.max():.1f}]')

spread_summary(base_data,    'Clean data (no outliers)')
print()
spread_summary(with_outlier, 'Data with 3 extreme outliers (150, 160, 155)')
print()
print('Observation:')
print('  Std jumped from ~5 to ~18 — a 3.5× increase due to just 3 outliers.')
print('  IQR barely changed — it is robust to extreme values.')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Std vs IQR: Robustness to Outliers', fontsize=12, fontweight='bold')

for ax, (data, title, color) in zip(axes, [
    (base_data,    'Clean data',                    COLORS['primary']),
    (with_outlier, 'With 3 outliers (150, 160, 155)', COLORS['danger']),
]):
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(),     color='black',           lw=2, linestyle='-',  label=f'Mean={data.mean():.1f}')
    ax.axvline(np.median(data), color=COLORS['success'], lw=2, linestyle='--', label=f'Median={np.median(data):.1f}')
    ax.axvspan(q1, q3, alpha=0.2, color=COLORS['accent'], label=f'IQR=[{q1:.1f},{q3:.1f}]')
    ax.set_title(f'{title}\nStd={data.std(ddof=1):.2f}  IQR={iqr:.2f}')
    ax.set_xlabel('Value'); ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


### 3.3 Coefficient of Variation (CV): Comparing Groups on Different Scales

Standard deviation is in the same unit as the data, making it impossible to
directly compare spread across groups with different means (e.g. height in cm
vs. weight in kg). The **Coefficient of Variation** solves this:

$$\text{CV} = \frac{\sigma}{\mu} \times 100\%$$

CV is dimensionless — a higher CV means *more relative variability*.


In [ ]:
# Compare variability across groups with different scales
groups = {
    'Height (cm)':   RNG.normal(170, 8,   size=300),
    'Weight (kg)':   RNG.normal(65,  10,  size=300),
    'Income (k¥)':   RNG.lognormal(4, 0.6, size=300),
    'Age (years)':   RNG.normal(35,  7,   size=300),
}

print(f'{"Group":<18}  {"Mean":>8}  {"Std":>8}  {"CV":>8}')
print('-' * 50)
for name, data in groups.items():
    mu  = data.mean()
    sig = data.std(ddof=1)
    cv  = sig / mu * 100
    print(f'{name:<18}  {mu:>8.2f}  {sig:>8.2f}  {cv:>7.1f}%')

print()
print('Interpretation:')
print('  Income has the highest CV (~60%) → relatively the most variable.')
print('  Height has the lowest CV (~5%)  → relatively the most uniform.')
print('  You cannot compare Std directly across these groups (different units),')
print('  but CV gives a fair comparison.')


---

## Part 4: Distribution Shape — Skewness and Kurtosis

Summary statistics (mean, std) describe location and spread, but they say nothing
about the *shape* of the distribution. Two additional statistics capture shape:

### 4.1 Skewness: Asymmetry

$$\text{Skewness} = \frac{1}{n}\sum\left(\frac{x_i - \bar{x}}{s}\right)^3$$

| Value | Shape | Typical examples |
|---|---|---|
| ≈ 0 | Symmetric | Height, exam scores |
| > 0 (positive/right skew) | Long **right** tail | Income, waiting times, counts |
| < 0 (negative/left skew) | Long **left** tail | Age at death, test max-scores |

**Rule of thumb:** |skewness| > 1 warrants attention; consider log-transform.


In [ ]:
# Simulate three shape scenarios
shapes = {
    'Symmetric\n(Normal, skew≈0)':      RNG.normal(loc=50, scale=10, size=2000),
    'Right-skewed\n(Log-normal, skew>0)': RNG.lognormal(mean=3.5, sigma=0.7, size=2000),
    'Left-skewed\n(reflected log-normal)': -RNG.lognormal(mean=3.5, sigma=0.7, size=2000) + 2*np.exp(3.5+0.7**2/2),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Skewness: Three Shapes', fontsize=13, fontweight='bold')

for ax, (name, data) in zip(axes, shapes.items()):
    skew = stats.skew(data)
    ax.hist(data, bins=50, density=True, color=COLORS['primary'], edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(),     color=COLORS['danger'],  lw=2, linestyle='-',  label=f'Mean={data.mean():.1f}')
    ax.axvline(np.median(data), color=COLORS['success'], lw=2, linestyle='--', label=f'Median={np.median(data):.1f}')
    ax.set_title(f'{name}\nskewness={skew:.2f}')
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Notice how mean > median for right-skewed data (long right tail pulls mean up),')
print('and mean < median for left-skewed data.')


### 4.2 Kurtosis: Tail Heaviness

$$\text{Excess\ Kurtosis} = \frac{1}{n}\sum\left(\frac{x_i - \bar{x}}{s}\right)^4 - 3$$

The −3 makes the Normal distribution the reference (excess kurtosis = 0).

| Excess kurtosis | Name | Shape | Examples |
|---|---|---|---|
| ≈ 0 | Mesokurtic | Normal-like tails | Heights |
| > 0 | Leptokurtic | **Fat tails** — extreme values more common | Financial returns, network degrees |
| < 0 | Platykurtic | **Thin tails** — values concentrated near centre | Uniform, bounded variables |

> **Why care?** Fat-tailed data violates normality assumptions in many statistical
> tests. Outliers are more frequent than expected.


In [ ]:
# Compare kurtosis across four distributions
kurt_examples = {
    'Normal\n(kurtosis=0)':           RNG.normal(0, 1, size=5000),
    'Uniform\n(kurtosis≈−1.2)':       RNG.uniform(-3, 3, size=5000),
    'Laplace\n(fat tails, kurtosis=3)': RNG.laplace(0, 1/np.sqrt(2), size=5000),
    't(df=3)\n(very fat tails)':       RNG.standard_t(df=3, size=5000),
}

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle('Kurtosis: Tail Heaviness Comparison', fontsize=13, fontweight='bold')

x_range = np.linspace(-5, 5, 300)
for ax, (name, data) in zip(axes, kurt_examples.items()):
    kurt = stats.kurtosis(data)   # excess kurtosis
    data_clip = np.clip(data, -6, 6)
    ax.hist(data_clip, bins=60, density=True, color=COLORS['primary'], edgecolor='white', alpha=0.75)
    # Overlay normal for reference
    ax.plot(x_range, stats.norm.pdf(x_range), color=COLORS['danger'],
            lw=1.5, linestyle='--', alpha=0.7, label='Normal ref.')
    ax.set_xlim(-6, 6)
    ax.set_title(f'{name}\nexcess kurtosis={kurt:.2f}')
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print('The Laplace and t(df=3) distributions have much heavier tails than the Normal.')
print('For t(df=3), extreme events (|x|>3) are about 3× more frequent than Normal.')


---

## Part 5: The EDA Visualisation Toolkit

Five plots cover 95% of univariate EDA needs. Know when to use each.

| Plot | Strength | Weakness |
|---|---|---|
| Histogram | Easy to interpret, shows density | Sensitive to bin width choice |
| KDE (kernel density) | Smooth; good for overlaying groups | Can extend past real data boundaries |
| Boxplot | Compact; great for many groups side-by-side | Hides bimodality |
| Violin | Combines KDE + boxplot | Can be misleading with small n |
| Q-Q plot | Direct normality test | Requires understanding of quantile theory |

### 5.1 Histogram: Bin Width Matters


In [ ]:
# Demonstrate how bin width changes the story
sample_data = np.concatenate([
    RNG.normal(30, 5, 300),
    RNG.normal(60, 8, 500),
])  # bimodal

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle('Histogram: Same Data, Different Bin Widths', fontsize=12, fontweight='bold')

for ax, bins in zip(axes, [5, 15, 40, 100]):
    ax.hist(sample_data, bins=bins, color=COLORS['primary'], edgecolor='white', alpha=0.8)
    ax.set_title(f'bins={bins}')
    ax.set_xlabel('Value'); ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

print('Rule of thumb for bin count:')
print('  Sturges: k = ⌈log₂(n) + 1⌉ =', int(np.ceil(np.log2(len(sample_data))+1)))
print('  Square root: k = ⌈√n⌉         =', int(np.ceil(np.sqrt(len(sample_data)))))
print('  Freedman-Diaconis (scipy default): automatic, data-driven')
print()
print('  → For EDA, try 2-3 bin counts. If bimodality appears at one setting')
print('    and disappears at another, report both and investigate.')


### 5.2 KDE: Smoother than a Histogram

A Kernel Density Estimate places a small "bump" (kernel) at each data point and
sums them into a smooth curve. It is ideal for overlaying two or more groups.


In [ ]:
group_a = RNG.normal(loc=50, scale=8,  size=200)
group_b = RNG.normal(loc=62, scale=10, size=200)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('KDE vs Histogram: Same Information, Different Form', fontsize=12, fontweight='bold')

# Left: histograms
axes[0].hist(group_a, bins=25, density=True, color=COLORS['primary'], alpha=0.6,
             edgecolor='white', label='Group A (μ=50)')
axes[0].hist(group_b, bins=25, density=True, color=COLORS['secondary'], alpha=0.6,
             edgecolor='white', label='Group B (μ=62)')
axes[0].set_title('Overlapping Histograms')
axes[0].set_xlabel('Score'); axes[0].set_ylabel('Density')
axes[0].legend()

# Right: KDE
sns.kdeplot(group_a, ax=axes[1], color=COLORS['primary'],   linewidth=2.5,
            fill=True, alpha=0.3, label='Group A (μ=50)')
sns.kdeplot(group_b, ax=axes[1], color=COLORS['secondary'], linewidth=2.5,
            fill=True, alpha=0.3, label='Group B (μ=62)')
axes[1].set_title('KDE — cleaner separation with overlapping groups')
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Density')
axes[1].legend()
plt.tight_layout()
plt.show()


### 5.3 Boxplot, Violin, and Strip Plot: Three Views of Group Comparison


In [ ]:
# Generate four groups with different shapes for comparison
n_per = 120
groups_df = pd.DataFrame({
    'value': np.concatenate([
        RNG.normal(50, 8,   n_per),
        RNG.lognormal(3.8, 0.4, n_per),
        RNG.normal(45, 15,  n_per),
        np.concatenate([RNG.normal(35,4,n_per//2), RNG.normal(65,4,n_per//2)]),
    ]),
    'group': (['Normal(50,8)'] * n_per +
              ['LogNormal'] * n_per +
              ['Wide(45,15)'] * n_per +
              ['Bimodal'] * n_per),
})

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Three Views of the Same Four Groups', fontsize=12, fontweight='bold')

palette = [COLORS['primary'], COLORS['secondary'], COLORS['accent'], COLORS['success']]

# Boxplot
sns.boxplot(data=groups_df, x='group', y='value', ax=axes[0], palette=palette)
axes[0].set_title('Boxplot\n(compact, shows IQR & outliers)')
axes[0].set_xlabel('Group'); axes[0].set_ylabel('Value')
axes[0].tick_params(axis='x', rotation=15)

# Violin
sns.violinplot(data=groups_df, x='group', y='value', ax=axes[1], palette=palette, inner='box')
axes[1].set_title('Violin plot\n(shows distribution shape)')
axes[1].set_xlabel('Group'); axes[1].set_ylabel('Value')
axes[1].tick_params(axis='x', rotation=15)

# Strip plot
sns.stripplot(data=groups_df, x='group', y='value', ax=axes[2],
              palette=palette, alpha=0.35, size=3, jitter=True)
axes[2].set_title('Strip plot\n(every individual point visible)')
axes[2].set_xlabel('Group'); axes[2].set_ylabel('Value')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print('The "Bimodal" group has two distinct clusters.')
print('  Boxplot: looks like a normal wide distribution — MISSES the bimodality.')
print('  Violin:  clearly shows two bumps — REVEALS the bimodality.')
print('  Strip:   individual points show two clusters — CONFIRMS bimodality.')
print()
print('→ Never use boxplot alone when you suspect non-unimodal distributions.')


---

## Part 6: Outlier Detection

An **outlier** is an observation far from the main body of the data.
Outliers can be:
- **Errors** (data entry mistake, sensor malfunction) → should be corrected or removed
- **Genuine extremes** (a real billionaire in an income survey) → keep and report
- **Interesting signals** (anomalous patient, fraud detection) → investigate

Two standard detection methods:

### 6.1 Z-score Method

$$z_i = \frac{x_i - \bar{x}}{s}$$

Flag observations with |z| > 3 (about 0.3% of normal data, by the 68-95-99.7 rule).

**Weakness:** the mean and std are *themselves* affected by outliers, so severe
outliers can inflate s enough to mask themselves ("masking effect").


In [ ]:
# Generate clean data and inject outliers
clean    = RNG.normal(loc=100, scale=15, size=200)
outliers = np.array([180, 185, 200, 10, -5])
mixed    = np.concatenate([clean, outliers])

# Z-score method
z_scores = (mixed - mixed.mean()) / mixed.std(ddof=1)
z_flags  = np.abs(z_scores) > 3

# Robust z-score: use median and MAD instead of mean and std
mad       = np.median(np.abs(mixed - np.median(mixed)))
z_robust  = 0.6745 * (mixed - np.median(mixed)) / mad   # scale factor → consistent with std
rz_flags  = np.abs(z_robust) > 3.5

print(f'=== Z-score method (|z| > 3) ===')
print(f'  Flagged {z_flags.sum()} points: {mixed[z_flags].round(1).tolist()}')
print()
print(f'=== Robust Z-score (|robust-z| > 3.5) ===')
print(f'  Flagged {rz_flags.sum()} points: {mixed[rz_flags].round(1).tolist()}')
print()
print('Note: standard z-score misses some outliers when the outliers inflate std.')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Z-score Outlier Detection', fontsize=12, fontweight='bold')

for ax, flags, title in [
    (ax1, z_flags,  'Standard z-score (threshold |z|>3)'),
    (ax2, rz_flags, 'Robust z-score (threshold |rz|>3.5)'),
]:
    ax.scatter(range(len(mixed)), mixed, s=15, alpha=0.5,
               c=[COLORS['danger'] if f else COLORS['primary'] for f in flags])
    for i, (val, flag) in enumerate(zip(mixed, flags)):
        if flag:
            ax.annotate(f'{val:.0f}', (i, val), textcoords='offset points',
                        xytext=(5, 5), fontsize=8, color=COLORS['danger'])
    ax.axhline(mixed.mean(),              color='black',           lw=1.5, linestyle='-',  label='Mean')
    ax.axhline(mixed.mean() + 3*mixed.std(ddof=1), color=COLORS['danger'], lw=1, linestyle=':', label='±3σ')
    ax.axhline(mixed.mean() - 3*mixed.std(ddof=1), color=COLORS['danger'], lw=1, linestyle=':')
    ax.set_title(f'{title}\n({flags.sum()} flagged, red dots)')
    ax.set_xlabel('Index'); ax.set_ylabel('Value')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 6.2 IQR Method (Tukey Fences)

$$\text{Lower fence} = Q1 - 1.5 \times \text{IQR}$$
$$\text{Upper fence} = Q3 + 1.5 \times \text{IQR}$$

Points outside these fences are plotted as individual dots in a boxplot.
This method is immune to masking because IQR is unaffected by outliers.

The 1.5 multiplier (Tukey's default) captures ~0.7% of normal data.
Using **3.0** (extreme outliers) reduces false positives.


In [ ]:
# IQR method — detailed walkthrough
data_iqr = np.concatenate([RNG.normal(100, 15, 300), [200, 210, -20, 220]])

q1, q3  = np.percentile(data_iqr, [25, 75])
iqr     = q3 - q1
fence_lo = q1 - 1.5 * iqr
fence_hi = q3 + 1.5 * iqr

flags_iqr = (data_iqr < fence_lo) | (data_iqr > fence_hi)

print(f'=== IQR Method ===')
print(f'  Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}')
print(f'  Lower fence = {q1:.2f} - 1.5×{iqr:.2f} = {fence_lo:.2f}')
print(f'  Upper fence = {q3:.2f} + 1.5×{iqr:.2f} = {fence_hi:.2f}')
print(f'  Flagged {flags_iqr.sum()} outliers: {data_iqr[flags_iqr].round(1).tolist()}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('IQR Outlier Detection (Tukey Fences)', fontsize=12, fontweight='bold')

# Boxplot showing outlier dots
ax1.boxplot(data_iqr, vert=True, patch_artist=True,
            boxprops=dict(facecolor=COLORS['primary'], alpha=0.5),
            medianprops=dict(color=COLORS['danger'], lw=2),
            flierprops=dict(marker='o', color=COLORS['danger'], markersize=7))
ax1.set_title('Boxplot (outliers shown as red dots)')
ax1.set_ylabel('Value')

# Manual fence visualisation
ax2.scatter(range(len(data_iqr)), data_iqr, s=15, alpha=0.4,
            c=[COLORS['danger'] if f else COLORS['primary'] for f in flags_iqr])
ax2.axhline(fence_hi, color=COLORS['danger'], lw=1.5, linestyle='--', label=f'Upper fence={fence_hi:.1f}')
ax2.axhline(fence_lo, color=COLORS['danger'], lw=1.5, linestyle='--', label=f'Lower fence={fence_lo:.1f}')
ax2.axhline(q1, color=COLORS['accent'],  lw=1, linestyle=':', label=f'Q1={q1:.1f}')
ax2.axhline(q3, color=COLORS['success'], lw=1, linestyle=':', label=f'Q3={q3:.1f}')
ax2.set_title('IQR fences — flagged points in red')
ax2.set_xlabel('Index'); ax2.set_ylabel('Value')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()


---

## Part 7: Why You Must Always Visualise — Anscombe's Quartet

In 1973, statistician Francis Anscombe constructed four datasets that are
**nearly identical in every summary statistic** yet look completely different
when plotted. This is one of the most important demonstrations in statistics.

> **Lesson:** Summary statistics are a compression of information.
> They can hide structure that is obvious in a plot.


In [ ]:
# Anscombe's Quartet — exact values
anscombe = {
    'I':   {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]},
    'II':  {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]},
    'III': {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]},
    'IV':  {'x': [8,8,8,8,8,8,8,19,8,8,8],
             'y': [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]},
}

# Compute summary stats for each
print(f'{"Dataset":<10} {"Mean X":>8} {"Mean Y":>8} {"Var X":>8} {"Var Y":>8} {"Corr(X,Y)":>10} {"R²":>6}')
print('-' * 65)
for name, d in anscombe.items():
    x, y = np.array(d['x']), np.array(d['y'])
    r, _ = stats.pearsonr(x, y)
    b1 = r * y.std(ddof=1) / x.std(ddof=1)
    b0 = y.mean() - b1 * x.mean()
    print(f'{name:<10} {x.mean():>8.2f} {y.mean():>8.2f} {x.var(ddof=1):>8.2f} '
          f'{y.var(ddof=1):>8.2f} {r:>10.3f} {r**2:>6.3f}')

print()
print('All four datasets have:')
print('  Nearly identical means, variances, correlation, and regression line.')
print('  → If you only looked at the numbers, you would conclude they are the same.')


In [ ]:
# Now visualise — the truth emerges
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Anscombe's Quartet — Same Statistics, Completely Different Data",
             fontsize=12, fontweight='bold')

colors_a = [COLORS['primary'], COLORS['secondary'], COLORS['accent'], COLORS['success']]

for ax, (name, d), color in zip(axes, anscombe.items(), colors_a):
    x, y = np.array(d['x']), np.array(d['y'])
    ax.scatter(x, y, color=color, s=60, zorder=3, edgecolors='white', linewidth=0.5)

    # Regression line
    b1 = stats.pearsonr(x, y)[0] * y.std(ddof=1) / x.std(ddof=1)
    b0 = y.mean() - b1 * x.mean()
    x_line = np.linspace(x.min()-0.5, x.max()+0.5, 100)
    ax.plot(x_line, b0 + b1*x_line, color='black', lw=1.5, linestyle='--', label='Fit')

    r, _ = stats.pearsonr(x, y)
    ax.set_title(f'Dataset {name}: r={r:.3f}, R²={r**2:.3f}')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)

plt.tight_layout()
plt.show()

print('Dataset I  : Linear relationship with random scatter — the "normal" case.')
print('Dataset II : Perfectly curved (quadratic) — linear fit is wrong model entirely.')
print('Dataset III: All points on a line EXCEPT one outlier — outlier drives the fit.')
print('Dataset IV : All X identical except one — correlation driven by a single lever point.')
print()
print('→ ALWAYS plot your data. Summary statistics lie by omission.')


---

## Part 8: Bivariate EDA — Two Variables Together

### 8.1 Scatter Plot: Your First Move with Two Continuous Variables


In [ ]:
# Simulate a dataset: test score vs study hours, with some noise
n = 300
study_hours = RNG.uniform(1, 8, n)
true_slope  = 7.5   # each extra hour → +7.5 points
true_int    = 30
noise       = RNG.normal(0, 10, n)
test_score  = true_int + true_slope * study_hours + noise

df_study = pd.DataFrame({'study_hours': study_hours, 'test_score': test_score})

print('Simulated data: test_score = 30 + 7.5 × study_hours + ε  (ε ~ N(0, 10))')
print(df_study.describe().round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Scatter Plot: Study Hours vs Test Score', fontsize=12, fontweight='bold')

# Basic scatter
axes[0].scatter(df_study.study_hours, df_study.test_score,
                color=COLORS['primary'], alpha=0.5, s=20)
# True line
x_line = np.linspace(1, 8, 100)
axes[0].plot(x_line, true_int + true_slope * x_line,
             color=COLORS['danger'], lw=2.5, label=f'True line: y=30+7.5x')
axes[0].set_xlabel('Study hours'); axes[0].set_ylabel('Test score')
axes[0].set_title('Raw scatter + true line')
axes[0].legend()

# With regression line estimated from data
from scipy.stats import linregress
slope, intercept, r_val, p_val, se = linregress(df_study.study_hours, df_study.test_score)
axes[1].scatter(df_study.study_hours, df_study.test_score,
                color=COLORS['primary'], alpha=0.5, s=20)
axes[1].plot(x_line, intercept + slope * x_line,
             color=COLORS['secondary'], lw=2.5, linestyle='--',
             label=f'Estimated: y={intercept:.1f}+{slope:.2f}x  (R²={r_val**2:.3f})')
axes[1].plot(x_line, true_int + true_slope * x_line,
             color=COLORS['danger'], lw=1.5, linestyle=':',
             label=f'True: y=30+7.5x')
axes[1].set_xlabel('Study hours'); axes[1].set_ylabel('Test score')
axes[1].set_title('Estimated regression vs. true line')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f'\nEstimated slope = {slope:.3f}  (true = 7.5)')
print(f'Estimated intercept = {intercept:.3f}  (true = 30)')
print(f'R² = {r_val**2:.3f}')


### 8.2 Pearson vs Spearman Correlation

| Correlation | What it measures | Requires | Robust to outliers? |
|---|---|---|---|
| **Pearson r** | Linear relationship | Continuous, roughly normal | No |
| **Spearman ρ** | Monotonic relationship (any direction) | Ordinal or continuous | Yes |

> **When to use Spearman:** skewed data, ordinal variables, when outliers are present,
> or when the relationship is monotone but non-linear.


In [ ]:
# Case A: Pearson and Spearman agree (linear, no outliers)
# Case B: They disagree (outlier distorts Pearson)
# Case C: Non-linear but monotone (Spearman wins)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Pearson vs Spearman Correlation: Three Scenarios', fontsize=12, fontweight='bold')

x_base = RNG.uniform(0, 10, 100)

# Case A: linear
y_A = 2 * x_base + RNG.normal(0, 2, 100)
pA, _ = stats.pearsonr(x_base, y_A)
sA, _ = stats.spearmanr(x_base, y_A)
axes[0].scatter(x_base, y_A, color=COLORS['primary'], alpha=0.6, s=20)
axes[0].set_title(f'A: Linear\nPearson r={pA:.3f}  Spearman ρ={sA:.3f}')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')

# Case B: one extreme outlier
x_B = np.append(x_base, 5)
y_B = np.append(2 * x_base + RNG.normal(0, 2, 100), 200)  # outlier at y=200
pB, _ = stats.pearsonr(x_B, y_B)
sB, _ = stats.spearmanr(x_B, y_B)
axes[1].scatter(x_B[:-1], y_B[:-1], color=COLORS['primary'], alpha=0.6, s=20, label='Data')
axes[1].scatter(x_B[-1:], y_B[-1:], color=COLORS['danger'], s=80, zorder=5, label='Outlier')
axes[1].set_title(f'B: With outlier (y=200)\nPearson r={pB:.3f}  Spearman ρ={sB:.3f}')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
axes[1].legend(fontsize=8)

# Case C: non-linear monotone (exponential)
x_C = RNG.uniform(0, 3, 100)
y_C = np.exp(x_C) + RNG.normal(0, 0.3, 100)
pC, _ = stats.pearsonr(x_C, y_C)
sC, _ = stats.spearmanr(x_C, y_C)
axes[2].scatter(x_C, y_C, color=COLORS['primary'], alpha=0.6, s=20)
axes[2].set_title(f'C: Non-linear monotone (y = eˣ)  Pearson r={pC:.3f}  Spearman ρ={sC:.3f}')
axes[2].set_xlabel('X'); axes[2].set_ylabel('Y')

plt.tight_layout()
plt.show()

print('Case A: Both agree — linear = monotone, no outliers.')
print('Case B: Pearson drops sharply, Spearman barely changes → outlier exposed.')
print('Case C: Spearman ≈ 1.0 (perfect monotone), Pearson misses the exponential curve.')


---

## Part 9: Multivariate Overview — Pair Plots and Heatmaps

When you have more than two variables, a **pair plot** (scatter matrix) and
a **correlation heatmap** give the fastest overview of all pairwise relationships.


In [ ]:
# Simulate a 4-variable dataset with planted correlations
n = 400

x1 = RNG.normal(50, 10, n)          # IQ score (synthetic)
x2 = 0.6 * x1 + RNG.normal(0, 8, n) # reading score (moderate + correlation with IQ)
x3 = 0.8 * x1 + RNG.normal(0, 5, n) # math score (high + correlation with IQ)
x4 = RNG.normal(30, 7, n)            # unrelated variable (study hours — independent)

df_multi = pd.DataFrame({
    'IQ':    x1,
    'Reading': x2,
    'Math':  x3,
    'Study_hrs': x4,
})

print('Planted correlations:')
print('  IQ ↔ Reading  : ~0.6 (moderate)')
print('  IQ ↔ Math     : ~0.8 (strong)')
print('  IQ ↔ Study_hrs: ~0.0 (independent)')
print()
print('Sample correlation matrix:')
print(df_multi.corr(method='pearson').round(3).to_string())


In [ ]:
# Pair plot
g = sns.pairplot(df_multi, diag_kind='kde', corner=True,
                 plot_kws={'alpha': 0.4, 's': 15, 'color': COLORS['primary']},
                 diag_kws={'color': COLORS['primary'], 'fill': True, 'alpha': 0.5})
g.figure.suptitle('Pair Plot — All Pairwise Scatter + Diagonal KDE', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
corr_matrix = df_multi.corr(method='pearson')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle = redundant
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, ax=ax, mask=mask,
            square=True, linewidths=0.5,
            annot_kws={'size': 11})
ax.set_title('Correlation Heatmap (lower triangle only)', fontsize=11)
plt.tight_layout()
plt.show()

print('The heatmap confirms what we planted:')
print('  IQ–Math:    r≈0.8 (dark red, strong positive)')
print('  IQ–Reading: r≈0.6 (orange, moderate positive)')
print('  Study–*:    r≈0.0 (white, no relationship)')


---

## Part 10: The Full EDA Pipeline — Applied to a Planted Dataset

We now assemble everything into a structured, reproducible EDA workflow.
We generate a dataset with **known structure** — groups, outliers, a
skewed variable, and planted correlations — then run the full pipeline to
verify we can recover every feature we planted.

**Planted structure:**
| Feature | What we planted |
|---|---|
| Two groups | Group A ~ Normal(65, 8), Group B ~ Normal(75, 10) |
| Skewed variable | Income ~ LogNormal(10.8, 0.5) |
| Outliers | 5 extreme values injected into Group A income |
| Correlation | age ↔ blood_pressure: r ≈ 0.55 |


In [ ]:
# ── Generate the dataset ──────────────────────────────────────────────────────
np.random.seed(42)
n_A, n_B = 150, 150

# Group A
age_A    = RNG.normal(35, 8, n_A).clip(18, 70).round(0)
bp_A     = 80 + 0.55 * (age_A - age_A.mean()) / age_A.std() * 15 + RNG.normal(0, 12, n_A)
income_A = RNG.lognormal(10.8, 0.5, n_A) / 1000  # in thousand yuan
score_A  = RNG.normal(65, 8, n_A)

# Group B
age_B    = RNG.normal(45, 10, n_B).clip(18, 75).round(0)
bp_B     = 80 + 0.55 * (age_B - age_B.mean()) / age_B.std() * 15 + RNG.normal(0, 14, n_B)
income_B = RNG.lognormal(11.0, 0.6, n_B) / 1000
score_B  = RNG.normal(75, 10, n_B)

# Inject outliers
income_A[:5] = [500, 600, 550, 480, 520]

df_full = pd.DataFrame({
    'group':        ['A'] * n_A + ['B'] * n_B,
    'age':          np.concatenate([age_A, age_B]),
    'blood_pressure': np.concatenate([bp_A, bp_B]),
    'income_k':     np.concatenate([income_A, income_B]),
    'test_score':   np.concatenate([score_A, score_B]),
})

print(f'Dataset shape: {df_full.shape}')
print(f'Groups: {df_full.group.value_counts().to_dict()}')
print()
print('=== Step 1: Data shape and types ===')
print(df_full.dtypes.to_string())
print()
print('=== Missing values ===')
print(df_full.isnull().sum().to_string())


In [ ]:
# ── Step 2: Univariate summaries ─────────────────────────────────────────────
print('=== Step 2: Summary statistics (all numeric variables) ===')
desc = df_full.describe().round(2)

# Add skewness and kurtosis
desc.loc['skewness'] = df_full.select_dtypes('number').apply(stats.skew).round(3)
desc.loc['kurtosis'] = df_full.select_dtypes('number').apply(stats.kurtosis).round(3)
print(desc.to_string())

print()
print('Observations:')
print(f'  income_k skewness = {stats.skew(df_full.income_k):.2f} → highly right-skewed (outliers)')
print(f'  test_score skewness = {stats.skew(df_full.test_score):.2f} → fairly symmetric')
print(f'  income_k max = {df_full.income_k.max():.0f}k → clearly our injected outliers')


In [ ]:
# ── Step 3: Full visualisation dashboard ─────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Full EDA Dashboard — Planted Dataset', fontsize=14, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# Row 0: univariate distributions
for col_idx, (var, label) in enumerate(zip(
    ['age', 'blood_pressure', 'income_k', 'test_score'],
    ['Age (years)', 'Blood Pressure', 'Income (k¥)', 'Test Score']
)):
    ax = fig.add_subplot(gs[0, col_idx])
    for grp, color in [('A', COLORS['primary']), ('B', COLORS['secondary'])]:
        subset = df_full[df_full.group == grp][var]
        sns.kdeplot(subset, ax=ax, color=color, fill=True, alpha=0.35,
                    linewidth=2, label=f'Group {grp}')
    ax.set_title(label); ax.set_xlabel(label); ax.set_ylabel('Density')
    if col_idx == 0: ax.legend(fontsize=8)

# Row 1: boxplots by group
for col_idx, var in enumerate(['age', 'blood_pressure', 'income_k', 'test_score']):
    ax = fig.add_subplot(gs[1, col_idx])
    sns.boxplot(data=df_full, x='group', y=var, ax=ax,
                palette=[COLORS['primary'], COLORS['secondary']])
    ax.set_xlabel('Group')

# Row 2, left: scatter age vs BP
ax_scatter = fig.add_subplot(gs[2, :2])
for grp, color in [('A', COLORS['primary']), ('B', COLORS['secondary'])]:
    sub = df_full[df_full.group == grp]
    ax_scatter.scatter(sub.age, sub.blood_pressure, s=15, alpha=0.5,
                       color=color, label=f'Group {grp}')
# Add overall regression line
slope_bp, int_bp, r_bp, *_ = linregress(df_full.age, df_full.blood_pressure)
x_arr = np.linspace(df_full.age.min(), df_full.age.max(), 100)
ax_scatter.plot(x_arr, int_bp + slope_bp * x_arr, color='black', lw=2,
                label=f'Regression r={r_bp:.2f}')
ax_scatter.set_xlabel('Age'); ax_scatter.set_ylabel('Blood Pressure')
ax_scatter.set_title('Age vs Blood Pressure (planted r≈0.55)')
ax_scatter.legend(fontsize=9)

# Row 2, right: heatmap
ax_heat = fig.add_subplot(gs[2, 2:])
corr_m = df_full.select_dtypes('number').corr()
mask_tri = np.triu(np.ones_like(corr_m, dtype=bool))
sns.heatmap(corr_m, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, ax=ax_heat,
            mask=mask_tri, square=True, linewidths=0.5,
            annot_kws={'size': 10})
ax_heat.set_title('Correlation matrix')

plt.show()


In [ ]:
# ── Step 4: Verify what we planted ───────────────────────────────────────────
from scipy.stats import ttest_ind, mannwhitneyu

print('=== Step 4: Verification ===')
print()

# Group means
for grp, planted in [('A', 65), ('B', 75)]:
    obs = df_full[df_full.group == grp]['test_score'].mean()
    print(f'Group {grp} test_score: planted μ={planted}, observed mean={obs:.2f}')

print()

# Age–BP correlation
r_age_bp, p_age_bp = stats.pearsonr(df_full.age, df_full.blood_pressure)
print(f'age–blood_pressure correlation: r={r_age_bp:.3f}  (planted ≈ 0.55)  p={p_age_bp:.4f}')

print()

# Income skewness
skew_income = stats.skew(df_full.income_k)
print(f'income_k skewness = {skew_income:.2f}  (expected >> 1 due to injected outliers)')

print()

# Outlier detection
q1, q3 = df_full.income_k.quantile([0.25, 0.75])
iqr_val = q3 - q1
outliers_detected = df_full[df_full.income_k > q3 + 1.5 * iqr_val]
print(f'Income outliers detected (IQR method): {len(outliers_detected)} rows')
print(f'  Outlier values: {sorted(outliers_detected.income_k.round(0).tolist(), reverse=True)[:8]}')
print(f'  (We injected 5 values at 480–600k; IQR method found {len(outliers_detected)} flags)')


---

## Summary

| Concept | Key takeaway |
|---|---|
| Mean vs Median | Use median (+ IQR) for skewed data; mean for symmetric |
| Std vs IQR | IQR is robust to outliers; std is not |
| CV | Use to compare relative variability across different-scale groups |
| Skewness | \|skew\| > 1 → consider log transform or non-parametric methods |
| Kurtosis | Excess > 0 → fat tails → normality tests may fail |
| Visualization | Always use histograms, KDE, boxplots before any test |
| Bimodality | Boxplot hides it; violin and strip plots reveal it |
| Outliers | Z-score (fast) vs IQR Tukey fences (robust); investigate before removing |
| Anscombe | Same statistics ≠ same data. **Always plot.** |
| Pearson vs Spearman | Spearman is robust to outliers and non-linear monotone relationships |
| Pair plot / heatmap | Fast multivariate overview before modelling |

### The EDA Checklist

```
1. Check shape: df.shape, df.dtypes, df.isnull().sum()
2. Summary stats: df.describe() + skewness + kurtosis
3. Univariate: histogram/KDE + boxplot for each variable
4. Outliers: boxplot whiskers, z-score, or IQR method
5. Bivariate: scatter plots for key pairs + correlation matrix
6. Group comparison: KDE or violin by category variable
7. Heatmap: correlation matrix for all numerical variables
8. Decide: transform skewed variables? Remove errors? Investigate outliers?
```

**Next lecture →** Lecture 03: Probability Distributions in Depth — PDF, CDF,
quantile functions, and how to choose the right distribution for your data.

---
*Python Statistical Analysis in Practice — Lecture 02*  
*Author: Jason JJ Li · PKU Institute of Population Research*
